In [1]:
import warnings
import gc

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from wan import WanI2V
from wan.modules.vae import WanVAE
from wan.modules.clip import CLIPModel
from wan.configs.wan_i2v_14B import i2v_14B
from wan.utils.utils import cache_video

from torchcodec.decoders import VideoDecoder
from torchvision import transforms

warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")

/local_scratch/gzappavi/wan_experiments/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/model.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)
/local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/model.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)


In [2]:
img = Image.open("examples/women_looking_at_each_other.jpg").convert("RGB")

target_size = (480, 832)

transform = transforms.Compose(
    [transforms.Resize(min(target_size)), transforms.CenterCrop(target_size)]
)
img = transform(img)
img.save("examples/transformed.jpg", quality=95)

In [3]:
import json


with open("examples/transformed.json") as f:
    annotations = json.load(f)

face_bboxes_dict = {
    shape["label"]: shape["points"]
    for shape in annotations["shapes"]
    if shape["shape_type"] == "rectangle"
}

face_bboxes = []
for name in ["red_dress_woman_face", "white_dress_woman_face"]:
    points = face_bboxes_dict[name]
    face_bboxes.append(points[0] + points[1])

In [4]:
def create_bbox_mask(bbox, image_size):
    """
    Creates a boolean mask for a bounding box.
    """
    top, left, bottom, right = bbox
    height, width = image_size

    y_coords, x_coords = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")

    # this looks counter-inutitive, but with meshgrid the origin is the top-left corner
    mask = (y_coords >= top) & (y_coords < bottom) & (x_coords >= left) & (x_coords < right)

    return mask


w, h = img.size
face_masks = torch.stack([create_bbox_mask(bbox, (h, w)) for bbox in face_bboxes])

In [5]:
frame_num = 81  # default
red_looks_white = torch.zeros(frame_num, dtype=bool)
red_looks_white[15:43] = True

white_looks_red = torch.zeros(frame_num, dtype=bool)
white_looks_red[34:69] = True

wlw = torch.stack([red_looks_white, white_looks_red])

In [6]:
wan_i2v = WanI2V(
    config=i2v_14B,
    checkpoint_dir="./weights/Wan2.1-I2V-14B-480P/",
    device_id=0,
    t5_cpu=True,
)

Loading checkpoint shards: 100%|██████████| 7/7 [01:05<00:00,  9.34s/it]


In [ ]:
# prompt = "Two young women, dressed in summer dresses, are walking and conversing in a vast, verdant field. The woman on the left has long, dark brown hair and is wearing a flowing, off-the-shoulder red dress with white patterns, looking towards her companion and smiling. The woman on the right has lighter, possibly reddish-blonde hair and is wearing a white sleeveless dress with small dark polka dots, also smiling and looking at her friend; both are wearing white sneakers. A narrow, grassy path is visible between rows of what appear to be young green bushes or crops, possibly berry bushes, stretching far into the background, with the rows creating a strong sense of perspective, converging towards the horizon under an overcast sky that suggests a soft, diffused light, contributing to the overall natural, serene, and friendly atmosphere of this relaxed interaction in an open agricultural landscape."

prompt = "Two smiling young women in summer dresses and white sneakers walk and converse in a vast, green field of uniform crop rows receding into the distance. The woman on the left wears a red, off-the-shoulder dress, while the woman on the right wears a white polka-dot dress. An overcast sky provides soft, diffused light over the serene agricultural landscape."
negative_prompt = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"

descr_list = [
    "woman on the left wearing a red, off-the-shoulder dress",
    "woman on the right wearing a white polka-dot dress",
]

link_text = " is looking at "

sampling_steps = 40
# sampling_steps = 2

timestep_bias_schedule = torch.zeros(sampling_steps, dtype=bool)
timestep_bias_schedule[:max(1, int(sampling_steps / 10))] = True

num_layers = wan_i2v.model.num_layers
blocks_bias_schedule = torch.zeros(num_layers, dtype=bool)
blocks_bias_schedule[:max(1, int(num_layers / 2))] = True

bias_kwargs = {
    "descr_list": descr_list,
    "link_text": link_text,
    "timestep_bias_schedule": timestep_bias_schedule,
    "blocks_bias_schedule": blocks_bias_schedule,
    "face_masks": face_masks,
    "wlw": wlw,
}

torch.cuda.synchronize()
gc.collect()
torch.cuda.empty_cache()
video, simil_masks_list = wan_i2v.generate(
    prompt,
    img,
    bias_kwargs,
    max_area=target_size[0] * target_size[1],
    n_prompt=negative_prompt,
    sampling_steps=sampling_steps,
    frame_num=frame_num,
)

KeyboardInterrupt: 

In [ ]:
video_file = cache_video(
    tensor=video.unsqueeze(0),
    save_file="example.mp4",
    fps=16,
    nrow=1,
    normalize=True,
    value_range=(-1, 1)
)
